# 06. YOLO11 객체 탐지

RTX 4090에서 사전 학습된 YOLO11n을 실행한다. 사진·영상에 있는 물체의 종류, 신뢰도, 위치를 확인하고 바운딩 박스를 그린다.

- 05번 이미지 생성과 독립 실행.
- COCO 80개 클래스의 객체 탐지를 지원하는 가중치 사용.
- 학습 실습이 아니므로 backward와 optimizer는 사용하지 않는다.
- 모델 가중치를 처음 받을 때 인터넷이 필요하다.

## 0. 패키지 설치

Jupyter에서 아래 셀을 처음 한 번 실행한다. 현재 커널 환경에 Ultralytics를 설치한다. 설치 직후 import 오류가 나면 커널을 재시작하고 다음 셀부터 실행한다.

In [ ]:
%pip install -q ultralytics

## 1. GPU·YOLO 모델·샘플 이미지 준비

YOLO11n은 nano 크기의 사전 학습 모델이다. 모델 파일은 처음 사용할 때 자동 다운로드된다.

In [ ]:
from pathlib import Path
import time

import torch
from PIL import Image
from IPython.display import display
from ultralytics import YOLO
from ultralytics.utils import ASSETS

assert torch.cuda.is_available(), "CUDA GPU가 있는 Jupyter 커널을 선택하세요."
device = 0
model = YOLO("yolo11n.pt")
output_dir = Path("runs/06_yolo_detection")
output_dir.mkdir(parents=True, exist_ok=True)
sample_image = ASSETS / "bus.jpg"
assert sample_image.is_file(), f"예제 이미지가 없습니다: {sample_image}"
print("GPU:", torch.cuda.get_device_name(0))
print("클래스 수:", len(model.names))
display(Image.open(sample_image))

## 2. 사진 탐지

- imgsz: YOLO 입력 이미지 크기 기준 (픽셀)
- conf: 결과에 남길 최소 신뢰도
- device=0: 첫 번째 NVIDIA GPU
- 결과 하나에는 클래스, 신뢰도, xyxy 박스 좌표가 들어 있다.

In [ ]:
result = model.predict(
    source=str(sample_image),
    imgsz=640,
    conf=0.25,
    device=device,
    verbose=False,
)[0]
print("원본 크기 (height, width):", result.orig_shape)
print("탐지 객체 수:", len(result.boxes))
print("xyxy shape:", tuple(result.boxes.xyxy.shape))

## 3. 탐지된 객체의 실제 값 확인

xyxy는 원본 이미지의 픽셀 좌표로, 왼쪽 위 (x1,y1)와 오른쪽 아래 (x2,y2)를 나타낸다.

In [ ]:
boxes = result.boxes
coords = boxes.xyxy.cpu().tolist()
confidences = boxes.conf.cpu().tolist()
class_ids = boxes.cls.int().cpu().tolist()

for i, (xyxy, score, class_id) in enumerate(zip(coords, confidences, class_ids), start=1):
    print(f"{i:2d} {result.names[class_id]:12s} conf={score:.3f} xyxy={[round(v, 1) for v in xyxy]}")

## 4. 박스를 그린 결과 저장

result.plot()은 박스와 클래스 이름을 이미지에 그려 준다. BGR 채널 순서를 RGB로 바꿔 표시한다.

In [ ]:
annotated = Image.fromarray(result.plot()[:, :, ::-1].copy())
display(annotated)
image_out = output_dir / "bus_detected.jpg"
annotated.save(image_out)
print("저장:", image_out.resolve())

## 5. 신뢰도 기준 바꿔보기

같은 사진을 서로 다른 conf 값으로 처리하고, 남는 박스의 개수를 비교한다.

In [ ]:
for threshold in [0.10, 0.25, 0.50]:
    r = model.predict(str(sample_image), imgsz=640, conf=threshold, device=device, verbose=False)[0]
    print(f"conf={threshold:.2f}, 탐지={len(r.boxes)}개")
    display(Image.fromarray(r.plot()[:, :, ::-1].copy()))

## 6. 자신의 사진으로 실행 (선택)

RunPod JupyterLab에 사진을 업로드하고 my_image에 그 파일 경로를 입력한다. 빈 문자열이면 건너뛴다.

In [ ]:
my_image = ""  # 예: "my_photo.jpg"
if my_image:
    assert Path(my_image).is_file(), f"파일 없음: {my_image}"
    my_result = model.predict(my_image, imgsz=640, conf=0.25, device=device, verbose=False)[0]
    my_annotated = Image.fromarray(my_result.plot()[:, :, ::-1].copy())
    display(my_annotated)
    my_out = output_dir / (Path(my_image).stem + "_detected.jpg")
    my_annotated.save(my_out)
    print("탐지:", len(my_result.boxes), "저장:", my_out.resolve())
else:
    print("사진을 업로드한 후 my_image를 설정하세요.")

## 7. 동영상에서 탐지 (선택)

MP4 등을 업로드하고 video_path에 파일 경로를 입력한다. stream=True를 사용하면 프레임별 탐지 결과를 한 번에 하나씩 받아 메모리 사용을 제한할 수 있다. save=True는 경계 상자를 그린 동영상 결과를 저장한다.

In [ ]:
video_path = ""  # 예: "input.mp4"
if video_path:
    assert Path(video_path).is_file(), f"파일 없음: {video_path}"
    frame_count = 0
    torch.cuda.synchronize()
    started = time.perf_counter()
    for frame_result in model.predict(
        video_path, device=device, imgsz=640, conf=0.25,
        stream=True, save=True, project=str(output_dir),
        name="video", exist_ok=True, verbose=False,
    ):
        frame_count += 1
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    print(f"{frame_count} frames / {elapsed:.2f} seconds")
    print("출력 폴더:", (output_dir / "video").resolve())
else:
    print("영상 경로를 설정하면 실행됩니다.")

## 8. YOLO11n과 YOLO11s 비교 (선택)

compare_models=True로 변경하면 더 큰 YOLO11s 가중치를 추가로 다운로드한다. 둘을 같은 이미지로 각각 워밍업 후 20회 실행해 시간을 비교한다. Python 호출·전처리·NMS 후처리까지 포함한 간이 벽시계 시간이다.

In [ ]:
compare_models = False
if compare_models:
    for weights in ["yolo11n.pt", "yolo11s.pt"]:
        detector = model if weights == "yolo11n.pt" else YOLO(weights)
        detector.predict(str(sample_image), imgsz=640, device=device, verbose=False)
        torch.cuda.synchronize()
        start = time.perf_counter()
        for _ in range(20):
            r = detector.predict(str(sample_image), imgsz=640, device=device, verbose=False)[0]
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
        print(f"{weights}: {elapsed * 1000 / 20:.1f} ms/이미지, 박스 {len(r.boxes)}개")
else:
    print("compare_models=True로 변경하면 비교합니다.")

## 실습에서 확인할 것

1. boxes.cls, boxes.conf, boxes.xyxy가 각각 무엇을 나타내는지.
2. confidence 기준을 바꿀 때 탐지 박스가 어떻게 변하는지.
3. 모델에 없는 종류의 물체를 사진에 넣으면 어떻게 되는지.
4. YOLO11n과 YOLO11s에서 속도와 탐지 결과가 어떻게 다른지.

참고: https://docs.ultralytics.com/usage/python/ / https://docs.ultralytics.com/models/yolo11/

실제 RTX 4090 실행 결과는 아직 검증하지 않았다. *.pt와 runs/는 .gitignore에 포함되어 있다.